In [ ]:
!nvidia-smi

In [ ]:
dataset_path = '/kaggle/input/datasets/adelekeadeniyan/nsl-alphanumeric-dataset/NSL_Dataset_Full_Combined/content/split_dataset'

In [ ]:
import os
print(os.listdir(dataset_path))

In [ ]:
import torch

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

In [ ]:
# 1. Install dependencies
!pip install -q --upgrade ultralytics albumentations opencv-python-headless tensorboard

In [ ]:
!pip install -q ultralytics

In [ ]:
print('\n--- Verifying ultralytics installation ---')
!pip show ultralytics
print('------------------------------------------')

In [ ]:
import os
import yaml

# Exact path printed by your output
dataset_path = '/kaggle/input/datasets/adelekeadeniyan/nsl-alphanumeric-dataset/content/split_dataset'

# Verify subdirectories directly
subfolders = os.listdir(dataset_path) if os.path.exists(dataset_path) else []
print("Directories found inside dataset:", subfolders)

# Auto-detect train/val/test subfolder naming conventions
train_folder = 'train/images' if 'train' in subfolders else 'images/train'
val_folder = 'val/images' if 'val' in subfolders else ('valid/images' if 'valid' in subfolders else 'images/val')
test_folder = 'test/images' if 'test' in subfolders else ('images/test' if 'images' in subfolders else val_folder)

yaml_content = {
    'path': dataset_path,
    'train': train_folder,
    'val': val_folder,
    'test': test_folder,
    'nc': 33,  # FIXED: Changed from 32 to 33
    'names': [
        '1', '2', '3', '4', '5', '6', '7', '8', '9',
        'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'k',
        'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u',
        'v', 'w', 'x', 'y'
    ]
}

with open('/kaggle/working/data.yaml', 'w') as f:
    yaml.dump(yaml_content, f)

print("data.yaml updated successfully!")

In [ ]:
from __future__ import annotations

import argparse
import os
import sys
import time
from pathlib import Path
import shutil
import yaml
import subprocess

_ultralytics_available = False
try:
    from ultralytics import YOLO
    _ultralytics_available = True
except ImportError:
    print("\n[CRITICAL ERROR] 'ultralytics' is not installed in the active environment. Please run: !pip install ultralytics\n")

KAGGLE_WORKING_DIR = Path("/kaggle/working")
DEFAULT_DATA_YAML = KAGGLE_WORKING_DIR / "data.yaml"
DEFAULT_MODEL = "yolo11n.pt"

# Root path of the uploaded Kaggle dataset
DATASET_ROOT = Path("/kaggle/input/datasets/adelekeadeniyan/nsl-alphanumeric-dataset/NSL_Dataset_Full_Combined/content/split_dataset")

CLASS_NAMES = [
    '1', '2', '3', '4', '5', '6', '7', '8', '9',
    'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'k',
    'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u',
    'v', 'w', 'x', 'y'
]


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Train YOLO11n model on Kaggle dataset with evaluation artifacts.",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    parser.add_argument(
        "--data",
        "-d",
        type=Path,
        default=DEFAULT_DATA_YAML,
        help="Path to YOLO dataset configuration file (data.yaml).",
    )
    parser.add_argument(
        "--model",
        "-m",
        type=str,
        default=DEFAULT_MODEL,
        help="Starting model weights or architecture (e.g., yolo11n.pt).",
    )
    parser.add_argument(
        "--epochs",
        "-e",
        type=int,
        default=50,
        help="Number of training epochs.",
    )
    parser.add_argument(
        "--imgsz",
        type=int,
        default=640,
        help="Training image resolution.",
    )
    parser.add_argument(
        "--batch",
        "-b",
        type=int,
        default=-1,
        help="Training batch size (-1 for AutoBatch).",
    )
    parser.add_argument(
        "--project",
        type=Path,
        default=KAGGLE_WORKING_DIR / "runs" / "detect",
        help="Ultralytics project output directory.",
    )
    parser.add_argument(
        "--name",
        type=str,
        default="train_yolo11n",
        help="Ultralytics run name for output artifacts.",
    )
    parser.add_argument(
        "--device",
        default=0,
        help="Target device ('0' for CUDA GPU, 'cpu', or leave empty for auto-detect).",
    )
    parser.add_argument(
        "--seed",
        type=int,
        default=42,
        help="Random seed for reproducible training.",
    )
    parser.add_argument(
        "--workers",
        type=int,
        default=8,
        help="Number of dataloader worker threads.",
    )
    parser.add_argument(
        "--cache",
        action="store_true",
        default=False,
        help="Cache images in RAM.",
    )
    parser.add_argument(
        "--no-amp",
        action="store_true",
        help="Disable Automatic Mixed Precision (AMP).",
    )
    return parser.parse_known_args()[0]


def format_time(seconds: float) -> str:
    """Format seconds into HHh MMm SSs."""
    seconds = int(seconds)
    hours, remainder = divmod(seconds, 3600)
    minutes, secs = divmod(remainder, 60)
    if hours > 0:
        return f"{hours}h {minutes:02d}m {secs:02d}s"
    return f"{minutes:02d}m {secs:02d}s"


def verify_thesis_artifacts(run_dir: Path) -> list[tuple[str, bool]]:
    """Check for existence of evaluation plots & metrics."""
    expected_artifacts = [
        ("weights/best.pt (Top Model Checkpoint)", run_dir / "weights" / "best.pt"),
        ("weights/last.pt (Final Epoch Checkpoint)", run_dir / "weights" / "last.pt"),
        ("F1_curve.png (F1 Score vs Confidence)", run_dir / "F1_curve.png"),
        ("PR_curve.png (Precision-Recall Curve)", run_dir / "PR_curve.png"),
        ("P_curve.png (Precision vs Confidence)", run_dir / "P_curve.png"),
        ("R_curve.png (Recall vs Confidence)", run_dir / "R_curve.png"),
        ("confusion_matrix.png (Confusion Matrix)", run_dir / "confusion_matrix.png"),
        ("confusion_matrix_normalized.png (Normalized Confusion Matrix)", run_dir / "confusion_matrix_normalized.png"),
        ("results.png (Loss & Metric Epoch Curves)", run_dir / "results.png"),
        ("results.csv (Numerical Loss/Metric Log)", run_dir / "results.csv"),
    ]

    return [(name, path.exists()) for name, path in expected_artifacts]


def locate_subpath(split_name: str) -> str:
    """Finds relative directory path for train/val/test images dynamically."""
    candidates = [
        DATASET_ROOT / split_name / "images",
        DATASET_ROOT / "images" / split_name,
        DATASET_ROOT / split_name,
    ]
    for path in candidates:
        if path.exists():
            return str(path.relative_to(DATASET_ROOT))
            
    # Fallback default
    return f"{split_name}/images"


def fix_and_get_data_yaml(target_yaml: Path) -> Path:
    """Fixes /kaggle/working/data.yaml to ensure valid relative paths for YOLO."""
    
    yaml_data = {
        'path': str(DATASET_ROOT.resolve()),
        'train': locate_subpath('train'),
        'val': locate_subpath('val'),
        'test': locate_subpath('test'),
        'nc': len(CLASS_NAMES),
        'names': CLASS_NAMES
    }

    target_yaml.parent.mkdir(parents=True, exist_ok=True)
    with open(target_yaml, 'w') as f:
        yaml.dump(yaml_data, f, default_flow_style=False, sort_keys=False)

    print(f"[INFO] Verified & formatted '{target_yaml.resolve()}' successfully.")
    print(f"[INFO] Dataset root set to: {yaml_data['path']}")
    print(f"[INFO] Train split path : {yaml_data['train']}")
    print(f"[INFO] Val split path   : {yaml_data['val']}")
    print(f"[INFO] Test split path  : {yaml_data['test']}")

    return target_yaml.resolve()


def run_yolo_training(args: argparse.Namespace) -> None:
    data_path = fix_and_get_data_yaml(args.data)
    run_dir = args.project.resolve() / args.name

    print("=" * 80)
    print(" YOLO11 Thesis Model Trainer & Evaluator (Kaggle Environment) ")
    print("=" * 80)
    print(f" Model Architecture     : {args.model}")
    print(f" Dataset Configuration  : {data_path}")
    print(f" Training Epochs        : {args.epochs}")
    print(f" Image Resolution       : {args.imgsz}x{args.imgsz}")
    print(f" Batch Size             : {args.batch}")
    print(f" Run Directory          : {run_dir}")
    print(f" Random Seed            : {args.seed}")
    print("-" * 80)

    print(f"[INFO] Initializing YOLO model '{args.model}'...")
    model = YOLO(args.model)

    start_time = time.time()

    print("\n[INFO] Starting model training...")
    results = model.train(
        data=str(data_path),
        epochs=args.epochs,
        imgsz=args.imgsz,
        batch=args.batch,
        project=str(args.project.resolve()),
        name=args.name,
        exist_ok=True,
        plots=True,
        save=True,
        seed=args.seed,
        workers=args.workers,
        device=args.device,
        cache=args.cache,
        amp=not args.no_amp,
    )

    elapsed_training = time.time() - start_time
    print(f"\n[SUCCESS] Model training completed in {format_time(elapsed_training)}!")

    best_weights_path = run_dir / "weights" / "best.pt"
    eval_model = YOLO(str(best_weights_path)) if best_weights_path.exists() else model

    print("\n" + "=" * 80)
    print(" THESIS EVALUATION: VALIDATION SET PERFORMANCE ")
    print("=" * 80)
    val_metrics = eval_model.val(data=str(data_path), split="val", imgsz=args.imgsz, plots=True)

    print("\n" + "=" * 80)
    print(" THESIS EVALUATION: TEST SET FINAL BENCHMARK ")
    print("=" * 80)
    test_metrics = eval_model.val(data=str(data_path), split="test", imgsz=args.imgsz, plots=True)

    print("\n" + "=" * 80)
    print(" MASTER'S / PHD THESIS PERFORMANCE SUMMARY ")
    print("=" * 80)
    
    header = f"{'Split':<12} | {'Precision (P)':<15} | {'Recall (R)':<15} | {'mAP@50':<15} | {'mAP@50-95':<15}"
    print(header)
    print("-" * 80)

    val_p = val_metrics.results_dict.get("metrics/precision(B)", 0.0)
    val_r = val_metrics.results_dict.get("metrics/recall(B)", 0.0)
    val_map50 = val_metrics.results_dict.get("metrics/mAP50(B)", 0.0)
    val_map = val_metrics.results_dict.get("metrics/mAP50-95(B)", 0.0)

    test_p = test_metrics.results_dict.get("metrics/precision(B)", 0.0)
    test_r = test_metrics.results_dict.get("metrics/recall(B)", 0.0)
    test_map50 = test_metrics.results_dict.get("metrics/mAP50(B)", 0.0)
    test_map = test_metrics.results_dict.get("metrics/mAP50-95(B)", 0.0)

    val_row = f"{'Validation':<12} | {val_p:<15.4f} | {val_r:<15.4f} | {val_map50:<15.4f} | {val_map:<15.4f}"
    test_row = f"{'Test Set':<12} | {test_p:<15.4f} | {test_r:<15.4f} | {test_map50:<15.4f} | {test_map:<15.4f}"

    print(val_row)
    print(test_row)
    print("=" * 80)

    print("\n" + "=" * 80)
    print(" GENERATED THESIS FIGURES & ARTIFACTS VERIFICATION ")
    print("=" * 80)
    artifacts = verify_thesis_artifacts(run_dir)
    for artifact_name, status in artifacts:
        symbol = "[✓]" if status else "[×]"
        print(f" {symbol} {artifact_name}")
    print("=" * 80 + "\n")


if __name__ == "__main__":
    if not _ultralytics_available:
        sys.exit(1)
    args = parse_args()
    run_yolo_training(args)

In [ ]:
import os
import shutil

folder_to_zip = '/kaggle/working/runs'
output_filename = '/kaggle/working/runs'
zip_file_path = f"{output_filename}.zip"

# 1. Check if the previous zip file exists and remove it
if os.path.exists(zip_file_path):
    os.remove(zip_file_path)
    print(f"Removed old archive: {zip_file_path}")

# 2. Verify the source runs folder exists and create the new zip
if os.path.exists(folder_to_zip):
    print(f"Zipping folder: {folder_to_zip}...")
    shutil.make_archive(output_filename, 'zip', folder_to_zip)
    print("Successfully created updated runs.zip archive!")
else:
    print(f"Error: Folder '{folder_to_zip}' not found.")

In [ ]:
from IPython.display import display, Javascript, Image as IPImage
from google.colab.output import eval_js if 'google.colab' in str(get_ipython()) else (None, None)
import cv2
import numpy as np
from ultralytics import YOLO
import base64

# Load updated YOLO model
model = YOLO('/kaggle/working/runs/detect/train_yolo11n/weights/best.pt')

# JavaScript for capturing a single frame from the browser webcam inside Kaggle UI
js_code = """
async function captureWebcam() {
    const div = document.createElement('div');
    const video = document.createElement('video');
    const button = document.createElement('button');
    
    button.textContent = '📸 Capture Hand Sign';
    button.style.padding = '10px 15px';
    button.style.margin = '10px 0';
    button.style.cursor = 'pointer';
    
    div.appendChild(video);
    div.appendChild(button);
    document.body.appendChild(div);
    
    const stream = await navigator.mediaDevices.getUserMedia({ video: true });
    video.srcObject = stream;
    await video.play();
    
    await new Promise((resolve) => button.onclick = resolve);
    
    const canvas = document.createElement('canvas');
    canvas.width = video.videoWidth;
    canvas.height = video.videoHeight;
    canvas.getContext('2d').drawImage(video, 0, 0);
    
    stream.getTracks().forEach(track => track.stop());
    div.remove();
    return canvas.toDataURL('image/jpeg', 0.85);
}
captureWebcam();
"""

print("Click '📸 Capture Hand Sign' in the frame below to run live detection:")

# Trigger JS camera interface
from IPython.display import HTML
display(HTML('''
<script>
''' + js_code + '''
</script>
'''))